# 001 - Fifa

In [77]:
import pandas as pd
import numpy as np
import polars as pl

In [79]:
df_matches = pd.read_csv('Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('Data/WorldCups.csv').rename(lambda col: col.lower())

## 🏆 Reto 1: El Filtro Histórico

**Objetivo:**  
Obtener todos los registros de los mundiales que se jugaron **después del año 2000**, ordenados del **más reciente al más antiguo**.

---

### 📌 Requerimientos

- Seleccionar **todos los datos** del dataset de Copas del Mundo.
- Filtrar únicamente los mundiales disputados **posteriores al año 2000**.
- Ordenar los resultados:
  - Primero el mundial **más reciente**
  - Luego hacia el **más antiguo**

---

### 🧠 Conceptos Clave
- `SELECT *`
- `WHERE`
- `ORDER BY`



```SQL
SELECT
    *
FROM worldcup.cups
WHERE year > 2000
ORDER BY year DESC;
```

In [83]:
df_c = df_cups.copy()

respuesta = (
    df_c[df_c['year'] > 2000].sort_values(by='year', ascending=False).reset_index(drop=True)
)

In [84]:
respuesta = pl_cups.filter(
    (pl.col('year') > 2000)
    ).sort('year', descending=True)

## 🏟️ El Reto: *Dominio Local*

**Objetivo:**  
Analizar qué tan fuerte es la localía en los partidos de fútbol, utilizando métricas de goles anotados por los equipos cuando juegan como locales.

---

### 📊 Fuente de Datos

- Tabla: `matches`  
- Nota: La tabla ya se encuentra con nombres de columnas en **minúsculas**.

---

### 📌 Requerimientos del Análisis

Tu misión es generar un reporte que cumpla con los siguientes puntos:

1. **Agrupación por equipo local**  
   Agrupa los datos utilizando la columna:
   - `home_team_name`

2. **Total de goles como local**  
   Calcula el total de goles anotados por cada equipo cuando juega de local:
   - Columna: `home_team_goals`
   - Métrica: **suma total**

3. **Máximo de goles en un solo partido**  
   Calcula el máximo de goles que un equipo ha anotado en un solo partido como local:
   - Columna: `home_team_goals`
   - Métrica: **máximo**

4. **Ordenamiento**  
   Ordena los resultados para identificar:
   - Los equipos con **mayor cantidad total de goles como local**

---

### 🧠 Pistas según la Herramienta

#### 🧩 SQL
- `GROUP BY`
- `SUM()`
- `MAX()`
- `ORDER BY`

#### 🐼 Pandas
```python
df.groupby('home_team_name').agg({
    'home_team_goals': ['sum', 'max']
})


```SQL
SELECT
    "Home Team Name",
    SUM("Home Team Goals") AS total_goals,
    MAX("Home Team Goals") AS max_goals
FROM worldcup.matches
GROUP BY "Home Team Name"
ORDER BY total_goals DESC;
```

In [88]:
df_m = df_matches.copy()

df_res = df_m.groupby('home team name').agg(
    total_goals = ('home team goals','sum'),
    max_goals = ('home team goals', 'max')
)

df_res = df_res.sort_values(by='total_goals', ascending=False).reset_index()

In [101]:
pl_res = (
    pl_matches.group_by('home team name') # Agrupamos por el texto
    .agg([
        # Operamos sobre los números (goles)
        pl.col('home team goals').sum().alias('total_goals'), 
        pl.col('home team goals').max().alias('max_goals')
    ])
    .sort('total_goals', descending=True)
)

## 🏆 Reto de Refuerzo: *El Mundial de 1994*

**Objetivo:**  
Analizar exclusivamente los partidos disputados durante el **Mundial de 1994**, enfocándonos en goles y asistencia por ciudad.

---

### 📊 Fuente de Datos

- Tabla: `matches`

---

### 📌 Requerimientos del Análisis

Tu misión es construir un reporte que cumpla con los siguientes pasos:

1. **Filtrado por año**  
   Selecciona únicamente los partidos correspondientes al año:
   - `year = 1994`

2. **Agrupación por ciudad**  
   Agrupa los registros utilizando la columna:
   - `city`

3. **Cálculos requeridos**

   - **Total de goles como local**  
     Calcula la suma de los goles anotados por los equipos locales:
     - Columna: `home_team_goals`

   - **Promedio de asistencia**  
     Calcula el promedio de público asistente a los partidos:
     - Columna: `attendance`

4. **Ordenamiento**  
   Ordena los resultados:
   - Desde la **ciudad con más goles**  
   - Hasta la **ciudad con menos goles**

---

### 🧠 Conceptos Clave
- `WHERE`
- `GROUP BY`
- `SUM()`
- `AVG()`
- `ORDER BY`


```SQL
SELECT
    city,
    SUM("Home Team Goals") AS total_goals,
    ROUND(AVG(attendance::integer),2) as avg_attendance
FROM worldcup.matches
WHERE year = 1994
GROUP BY city
order by total_goals DESC;
```

In [107]:
df_m = df_matches.copy()

df_m['attendace'] = pd.to_numeric(df_m['attendance'], errors='coerce')

res_m = (
    df_m[df_m['year'] == 1994]
    .groupby('city')
    .agg(
    total_goals = ('home team goals','sum'),
    avg_attendance = ('attendance','mean')
))

res_m['avg_attendance'] = res_m['avg_attendance'].round(2)

res_m = res_m.sort_values(by='total_goals', ascending=False).reset_index()

res_m

,city,total_goals,avg_attendance
0,San Francisco,14.0,81737.33
1,Los Angeles,12.0,92600.88
2,Dallas,11.0,58692.00
3,Chicago,10.0,62545.00
4,Boston,9.0,54021.67
5,Orlando,7.0,61265.80
6,New York/New Jersey,7.0,73689.71
7,Washington Dc,7.0,52309.00
8,Detroit,6.0,70899.50


In [111]:
pl_res = (
    pl_matches
    .filter(pl.col('year')==1994)
    .group_by('city')
    .agg([
        pl.col('home team goals').sum().alias('total_goals'),
        pl.col('attendance')
        .cast(pl.Int64, strict=False)
        .mean()
        .round(2)
        .alias('avg_attendance')
    ])
    .sort('total_goals', descending=True)
)

## 🏆 Reto de Refuerzo: *Duelo de Gigantes*

**Contexto:**  
Analizar el desempeño de los equipos cuando juegan como **visitantes**, enfocándonos únicamente en los dos países más icónicos de Sudamérica: **Brazil** y **Argentina**.

---

### 📊 Fuente de Datos

- Tabla: `matches`

---

### 📌 Requerimientos del Análisis

Tu misión es generar un reporte que cumpla con los siguientes pasos:

1. **Filtrado por equipo visitante**  
   Selecciona únicamente los partidos donde:
   - `away_team_name` sea **'Brazil'** o **'Argentina'**

2. **Agrupación por equipo visitante**  
   Agrupa los datos utilizando la columna:
   - `away_team_name`

3. **Cálculos requeridos**

   - **Total de goles como visitante**  
     Calcula la suma de los goles anotados por cada equipo cuando juega como visitante:
     - Columna: `away_team_goals`

   - **Cantidad de partidos jugados como visitante**  
     Calcula el número total de partidos disputados como visitante:
     - Puede realizarse mediante un **conteo de filas**

4. **Ordenamiento**  
   Ordena los resultados:
   - De **mayor a menor** según el **total de goles anotados**

---

### 🧠 Conceptos Clave
- `WHERE`
- Operadores lógicos (`OR`, `IN`)
- `GROUP BY`
- `SUM()`
- `COUNT()`
- `ORDER BY`


```SQL
SELECT
    "Away Team Name",
    COUNT("Away Team Name"),
    SUM("Away Team Goals") AS total_away_goals
FROM worldcup.matches
WHERE LOWER(TRIM("Away Team Name")) in ('brazil','argentina')
GROUP BY "Away Team Name"
ORDER BY total_away_goals DESC;
```

In [123]:
df_m = df_matches.copy()

df_m['away team name'] = df_m['away team name'].fillna('').str.strip().str.lower()

away_team = df_m[
    df_m['away team name'].isin(['brazil','argentina'])
]

res_m = away_team.groupby('away team name').agg(
    total_matches = ('away team goals', 'count'),
    total_away_goals=('away team goals','sum')
).reset_index().sort_values(by='total_away_goals', ascending=False)

res_m



,away team name,total_matches,total_away_goals
1,brazil,26,45.0
0,argentina,27,22.0


In [122]:
pl_res = (
    pl_matches
    .filter(
        pl.col('away team name')
        .str.strip_chars()
        .str.to_lowercase()
        .is_in(['brazil','argentina'])
    )
    .group_by('away team name')
    .agg([
        pl.len().alias('total matches'),
        pl.col('away team goals').sum().alias('total away goals')
    ])
    .sort('total away goals', descending=True)
)

pl_res

away team name,total matches,total away goals
str,u32,i64
"""Brazil""",26,45
"""Argentina""",27,22


## 🏆 El Desafío Final: *Análisis de Estadios Europeos*

Queremos saber cómo les va a los equipos en las **sedes europeas**, pero el dataset tiene un detalle importante:  
la columna `city` a veces contiene **espacios extra** o **variaciones en mayúsculas/minúsculas**.

---

### 🎯 Tu misión

#### 🔍 Filtrar
Queremos **solo** los partidos jugados en las siguientes ciudades:

- Paris  
- Lyon  
- Bordeaux  
- Marseille  

> ⚠️ Usa lógica de limpieza para que no importe si aparece como `" paris"`, `"PARIS"` o `"Paris "`.

---

#### 🔄 Transformar
- La columna `attendance` (asistencia) viene como **texto**.
- Debes convertirla a **número**.
- Si hay errores durante la conversión, conviértelos en **valores nulos (`NaN`)**.

---

#### 📊 Agrupar
Agrupa los datos por la columna:

- `city`

---

#### 🧮 Calcular
Para cada ciudad, calcula:

- El **promedio de goles** del equipo local (`home team goals`).
- El **promedio de asistencia** (`attendance`), **redondeado a 0 decimales**.

---

#### 🔽 Ordenar
- Ordena el resultado **de mayor a menor** según el **promedio de asistencia**.


```SQL
SELECT
    city,
    ROUND(AVG("Away Team Goals"),2) AS avg_away_goals,
    ROUND(AVG(attendance::integer),2) AS avg_attendence
FROM worldcup.matches
WHERE LOWER(TRIM(city)) in ('paris','lyon','bordeaux','marseille')
group by city
ORDER BY avg_attendence DESC;
```

In [142]:
df_m = df_matches.copy()

df_m['city'] = df_m['city'].str.strip().str.lower()
df_m['attendance'] = pd.to_numeric(df_m['attendance'], errors = 'coerce')

filter_city = df_m[
    df_m['city'].isin(['paris','lyon','bordeaux','marseille'])
].reset_index(drop=True)

res_m = filter_city.groupby('city').agg(
    avg_away_goals = ('away team goals','mean'),
    avg_attendence = ('attendance', 'mean')
).reset_index()

respuesta = res_m.sort_values(by='avg_attendence', ascending=False)

respuesta[['avg_away_goals','avg_attendence']] = respuesta[['avg_away_goals','avg_attendence']].round(2)

respuesta

,city,avg_away_goals,avg_attendence
1,lyon,1.83,39100.00
2,paris,0.89,37797.44
0,bordeaux,1.33,26995.78


In [146]:
pl_res = (
    pl_matches
    .filter(
        pl.col('city').str.strip_chars().str.to_lowercase().is_in(['paris','lyon','bordeaux','marseille'])
    )
    .group_by('city')
    .agg([
        pl.col('away team goals').mean().round(2).alias('avg_away_goals'),
        pl.col('attendance').cast(pl.Int64, strict=False).mean().round(2).alias('avg_attendence')
    ])
    .sort('avg_attendence', descending=True)
)

pl_res

city,avg_away_goals,avg_attendence
str,f64,f64
"""Lyon """,1.83,39100.0
"""Paris """,0.89,37797.44
"""Bordeaux """,1.33,26995.78


## 🏆 Desafío 1 (Nivel 3): *El Mapa de los Ganadores*

### 🧩 El problema
La tabla `cups` contiene el nombre del **ganador** (`winner`), pero **no indica en qué estadio** se jugó la final.  
Esa información del estadio se encuentra en la tabla `matches`.

---

### 🎯 Tu misión
Realiza una **unión (JOIN)** entre `cups` y `matches` para mostrar:

- 📅 El **año** (`year`)
- 🌍 El **país ganador** (`winner`)
- 🏟️ El **nombre del estadio** (`stadium`)

---

### ⚠️ Condición especial
- La tabla `matches` contiene **muchos partidos por año**.
- La tabla `cups` tiene **solo un ganador por año**.

👉 Por lo tanto, debes **filtrar los partidos** para quedarte **solo con la final**, es decir, aquellos registros donde:

```sql
stage = 'Final'


```SQL
SELECT
    c.year,
    c.winner,
    m.stadium
FROM worldcup.cups AS c
INNER JOIN worldcup.matches m on c.year = m.year
WHERE LOWER(TRIM(m.stage)) = 'final';
```

In [148]:
df_m = df_matches.copy()
df_c = df_cups.copy()

df_merge = df_c.merge(
    df_m,
    on='year',
    how='inner'
)

res = df_merge[
    df_merge['stage'].str.strip().str.lower() == 'final'
].reset_index()

res_f = res[['year','winner','stadium']]

In [152]:
pl_join = pl_cups.join(
    pl_matches,
    on='year',
    how='inner'
).filter(
    pl.col('stage').str.strip_chars().str.to_lowercase() == 'final'
).select(['year','winner','stadium'])

## 🏆 Desafío 2 (Nivel 3): *Efecto Localía*

### 🎯 Objetivo
Queremos analizar si los países **anfitriones del Mundial (Hosts)** realmente **anotan más goles cuando juegan en casa**.

---

### 🧩 El problema
Tenemos la información repartida en dos tablas:

1. La tabla `cups` contiene:
   - `country` → el **país anfitrión** del Mundial.

2. La tabla `matches` contiene:
   - `home_team_name` → el **equipo local** en cada partido.
   - `home_team_goals` → los **goles anotados por el equipo local**.

---

### 🚀 Tu misión
Realiza una **unión (JOIN)** entre las tablas `cups` y `matches` para determinar:

- Cuántos **goles anotó cada país anfitrión**
- **Solo** en los partidos donde:
  - El país fue **anfitrión del Mundial**
  - Y jugó como **equipo local** (`home_team_name`)

---

### 🧠 Pistas clave para el análisis
- Une las tablas usando el **año del Mundial (`year`)**.
- Filtra los partidos donde:
  - `home_team_name = country`
- Agrupa los resultados por **país anfitrión**.
- Suma los goles usando `home_team_goals`.


```SQL
SELECT
    c.country AS host,
    SUM("Home Team Goals") AS total_home_goals
FROM worldcup.cups AS c
INNER JOIN worldcup.matches m on c.year = m.year
WHERE c.country = m."Home Team Name"
GROUP BY c.country
ORDER BY total_home_goals DESC;
```

In [173]:
df_c = df_cups.copy()
df_m = df_matches.copy()

df_merge = df_c.merge(
    df_m,
    on='year',
    how='inner'
)

df_host = df_merge[
    df_merge['country'] == df_merge['home team name']
]

res_host = df_host.groupby('country').agg(
    total_home_goals = ('home team goals', 'sum')
).reset_index()


respuesta = res_host[['country','total_home_goals']].sort_values(by='total_home_goals', ascending=False)

respuesta = respuesta.rename(columns={'country':'host'})


In [172]:
pl_join = pl_cups.join(
    pl_matches,
    on='year',
    how = 'inner'
).filter(
    pl.col('country') == pl.col('home team name')
)

pl_res = (pl_join.group_by('country').agg(
    pl.col('home team goals').sum().alias('total_home_goals')
).select(['country','total_home_goals'])
.rename({'country':'host'})
.sort('total_home_goals', descending=True)
)

## 🏆 Desafío Final (Nivel 3): *La Efectividad del Campeón*

### 🎯 Objetivo
Queremos analizar **qué tan efectivo fue el campeón del Mundial en la Final**, midiendo:

- Cuántos **goles anotó el campeón** en ese partido.
- Qué **porcentaje del total de goles de la Final** representaron esos goles.

---

### 📊 Datos necesarios

#### Tabla `cups`
- `year` → Año del Mundial  
- `winner` → País campeón

#### Tabla `matches`
- `year` → Año del partido  
- `stage` → Fase del torneo  
- `home_team_name` → Equipo local  
- `away_team_name` → Equipo visitante  
- `home_team_goals` → Goles del equipo local  
- `away_team_goals` → Goles del equipo visitante  

---

### 🧠 Lógica del análisis

1. **Unir las tablas**
   - Realiza un `JOIN` entre `cups` y `matches` usando la columna `year`.

2. **Filtrar**
   - Quédate únicamente con los partidos donde:
     - `stage = 'Final'`

3. **Identificar los goles del campeón**
   - Crea una nueva columna llamada `champion_goals` que:
     - Tome `home_team_goals` si `home_team_name = winner`
     - Tome `away_team_goals` si `away_team_name = winner`
   - Puedes usar:
     - `CASE WHEN` en SQL
     - `np.where()` en Pandas
     - `pl.when().then().otherwise()` en Polars

4. **Calcular el total de goles del partido**
   - Suma:
     - `home_team_goals + away_team_goals`

5. **Calcular el porcentaje de efectividad**
   - `(champion_goals / total_goals) * 100`

---

### 🧩 Resultado esperado
Para cada Final del Mundial deberías poder ver:

- Año del Mundial  
- Campeón  
- Goles del campeón en la Final  
- Total de goles del partido  
- Porcentaje de goles aportados por el campeón  


```SQL
SELECT
    c.year,
    c.winner,
    CASE
        WHEN m."Home Team Name" = c.winner THEN m."Home Team Goals"
        WHEN m."Away Team Name" = c.winner THEN m."Away Team Goals"
        ELSE 0
    END AS champion_goals,
    (m."Home Team Goals" + m."Away Team Goals") AS total_goals,
    ROUND(
        CASE
            WHEN (m."Home Team Goals" + m."Away Team Goals") > 0 THEN
                (
                    CASE
                        WHEN m."Home Team Name" = c.winner THEN m."Home Team Goals"
                        WHEN m."Away Team Name" = c.winner THEN m."Away Team Goals"
                        ELSE 0
                    END::numeric
                    /
                    (m."Home Team Goals" + m."Away Team Goals")
                ) * 100
            ELSE 0
        END,
        2
    ) AS champion_goal_percentage
FROM worldcup.cups c
JOIN worldcup.matches m
    ON c.year = m.year
WHERE m.stage = 'Final';
```


In [178]:
df_merge = df_cups.merge(df_matches, on='year', how='inner')

df_final = df_merge[df_merge['stage'].str.strip().str.lower() == 'final'].copy()

df_final['champion_goals'] = np.where(
    df_final['home team name'] == df_final['winner'],
    df_final['home team goals'],
    df_final['away team goals']
)

df_final['total_goals'] = df_final['home team goals'] + df_final['away team goals']

df_final['champion_goal_percentage'] = (
    (df_final['champion_goals'] / df_final['total_goals']) * 100
).fillna(0).round(2)

res_final = df_final[['year','winner','champion_goals','total_goals','champion_goal_percentage']]

res_final = res_final.sort_values('year')

res_final

,year,winner,champion_goals,total_goals,champion_goal_percentage
17,1930,Uruguay,4.0,6.0,66.67
34,1934,Italy,2.0,3.0,66.67
52,1938,Italy,4.0,6.0,66.67
100,1954,Germany FR,3.0,5.0,60.00
135,1958,Brazil,5.0,7.0,71.43
167,1962,Brazil,3.0,4.0,75.00
199,1966,England,4.0,6.0,66.67
231,1970,Brazil,4.0,5.0,80.00
269,1974,Germany FR,2.0,3.0,66.67
307,1978,Argentina,3.0,4.0,75.00


In [181]:
pl_res = (
    pl_cups.join(pl_matches, on='year', how='inner')
    # 1. Filtro (equivalente al WHERE)
    .filter(pl.col('stage').str.strip_chars().str.to_lowercase() == 'final')
    
    # 2. Creamos los goles del campeón y el total (equivalente al CASE WHEN y suma)
    .with_columns([
        pl.when(pl.col('home team name') == pl.col('winner'))
        .then(pl.col('home team goals'))
        .otherwise(pl.col('away team goals'))
        .alias('champion_goals'),
        
        (pl.col('home team goals') + pl.col('away team goals')).alias('total_goals')
    ])
    
    # 3. Calculamos el porcentaje y el promedio histórico (Window Function)
    .with_columns([
        # Porcentaje: usamos fill_nan(0) para evitar errores si total_goals es 0
        ((pl.col('champion_goals') / pl.col('total_goals')) * 100)
        .round(2)
        .fill_nan(0)
        .alias('champion_goal_percentage'),
        
        # Window Function: Promedio de goles del campeón en toda la historia
        pl.col('champion_goals').mean().alias('avg_historic_goals')
    ])
    
    # 4. Seleccionamos y ordenamos
    .select([
        'year', 
        'winner', 
        'champion_goals', 
        'total_goals', 
        'champion_goal_percentage',
        'avg_historic_goals'
    ])
    .sort('year')
)

pl_res

year,winner,champion_goals,total_goals,champion_goal_percentage,avg_historic_goals
i64,str,i64,i64,f64,f64
1930,"""Uruguay""",4,6,66.67,2.5
1934,"""Italy""",2,3,66.67,2.5
1938,"""Italy""",4,6,66.67,2.5
1954,"""Germany FR""",3,5,60.0,2.5
1958,"""Brazil""",5,7,71.43,2.5
…,…,…,…,…,…
2002,"""Brazil""",2,2,100.0,2.5
2006,"""Italy""",1,2,50.0,2.5
2010,"""Spain""",1,1,100.0,2.5
